# ConvNeXtBase — TensorFlow / Keras

Pure-CNN that adopts Vision Transformer design principles: depthwise 7×7 conv, LayerNorm, GELU, inverted bottleneck MLP, and per-channel LayerScale.  
**dims=[128, 256, 512, 1024]  |  depths=[3, 3, 27, 3]  |  ~89M params  |  Input: 224×224**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import roc_auc_score

print("TF:", tf.__version__)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


class LayerScale(layers.Layer):
    """Per-channel learnable scale, initialised to a small constant (1e-6)."""

    def __init__(self, dim, init_value=1e-6, **kwargs):
        super().__init__(**kwargs)
        self._dim        = dim
        self._init_value = init_value

    def build(self, input_shape):
        self.gamma = self.add_weight(
            shape       = (self._dim,),
            initializer = tf.keras.initializers.Constant(self._init_value),
            trainable   = True,
            name        = "gamma",
        )

    def call(self, x):
        return x * self.gamma

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"dim": self._dim, "init_value": self._init_value})
        return cfg


def convnext_block(x, dim, layer_scale_init=1e-6):
    """ConvNeXt block: DWConv7x7 - LN - Dense(4d) - GELU - Dense(d) - LayerScale - Add."""
    shortcut = x
    # Depthwise 7x7 conv (large receptive field, channel-wise)
    x = layers.DepthwiseConv2D(7, padding="same", use_bias=True)(x)
    # Layer normalisation (applied across channels at each spatial position)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    # Inverted bottleneck MLP via Dense on the channel axis
    x = layers.Dense(4 * dim)(x)
    x = layers.Activation("gelu")(x)
    x = layers.Dense(dim)(x)
    # Per-channel learnable scale (stabilises training of deep networks)
    x = LayerScale(dim, init_value=layer_scale_init)(x)
    # Residual connection
    x = layers.Add()([shortcut, x])
    return x

def build_convnext_base(num_classes=1000, input_shape=(224, 224, 3)):
    """ConvNeXt-Base: dims=[128, 256, 512, 1024], depths=[3, 3, 27, 3]."""
    dims   = [128, 256, 512, 1024]
    depths = [3, 3, 27, 3]
    inputs = keras.Input(shape=input_shape)

    # Stem: patchify with 4x4 non-overlapping conv, stride 4 -> 56x56
    x = layers.Conv2D(dims[0], 4, strides=4, padding="same")(inputs)
    x = layers.LayerNormalization(epsilon=1e-6)(x)

    # Four stages; each stage except the first is preceded by downsampling
    for i in range(4):
        if i > 0:
            # Downsampling: LN + 2x2 stride-2 conv -> halve spatial dims
            x = layers.LayerNormalization(epsilon=1e-6)(x)
            x = layers.Conv2D(dims[i], 2, strides=2, padding="same")(x)
        for _ in range(depths[i]):
            x = convnext_block(x, dims[i])

    # Classification head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return keras.Model(inputs, outputs, name="ConvNeXtBase")


if __name__ == "__main__":
    model = build_convnext_base()
    model.summary()

In [ ]:
model = build_convnext_base(num_classes=10)
model.summary()

In [ ]:
dummy = tf.random.normal([2, 224, 224, 3])
out   = model(dummy, training=False)
print("Output shape:", out.shape)

In [ ]:
total = sum(p.numpy().size for p in model.trainable_weights)
print(f"Trainable params: {total:,}")

In [ ]:
layer_info = [
    (l.name, l.__class__.__name__, l.count_params())
    for l in model.layers
]
for lname, typ, p in layer_info[-20:]:
    print(f"{lname:45s}  {typ:30s}  {p:>10,}")

## Training

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

BATCH     = 32
IMG_SIZE  = (224, 224)
TRAIN_DIR = "dataset/train"
VAL_DIR   = "dataset/val"

train_gen = ImageDataGenerator(
    rescale            = 1.0 / 255,
    rotation_range     = 20,
    width_shift_range  = 0.1,
    height_shift_range = 0.1,
    horizontal_flip    = True,
    zoom_range         = 0.1,
)
val_gen = ImageDataGenerator(rescale=1.0 / 255)

train_data = train_gen.flow_from_directory(
    TRAIN_DIR,
    target_size = IMG_SIZE,
    batch_size  = 32,
    class_mode  = "categorical",
)
val_data = val_gen.flow_from_directory(
    VAL_DIR,
    target_size = IMG_SIZE,
    batch_size  = 32,
    class_mode  = "categorical",
    shuffle     = False,
)

NUM_CLASSES = len(train_data.class_indices)
print("Classes:", NUM_CLASSES)

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau

model = build_convnext_base(num_classes=NUM_CLASSES)
model.compile(
    optimizer = keras.optimizers.Adam(1e-3),
    loss      = "categorical_crossentropy",
    metrics   = ["accuracy"],
)

callbacks = [
    ModelCheckpoint(
        "convnext_base_best.keras",
        monitor        = "val_accuracy",
        save_best_only = True,
        verbose        = 1,
    ),
    ReduceLROnPlateau(
        monitor  = "val_loss",
        factor   = 0.1,
        patience = 5,
        min_lr   = 1e-7,
        verbose  = 1,
    ),
]

history = model.fit(
    train_data,
    validation_data = val_data,
    epochs          = 30,
    callbacks       = callbacks,
)

## Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["accuracy"],     label="train")
axes[0].plot(history.history["val_accuracy"], label="val")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history.history["loss"],     label="train")
axes[1].plot(history.history["val_loss"], label="val")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

## Inference

In [ ]:
from tensorflow.keras.preprocessing.image import load_img, img_to_array

CLASS_NAMES = list(train_data.class_indices.keys())


def predict_image(img_path, model, class_names, img_size=(224, 224)):
    img   = load_img(img_path, target_size=img_size)
    x     = img_to_array(img) / 255.0
    x     = np.expand_dims(x, axis=0)
    probs = model.predict(x, verbose=0)[0]
    idx   = np.argmax(probs)
    print(f"Prediction: {class_names[idx]}  ({probs[idx]*100:.1f}%)")
    return class_names[idx], probs


# predict_image("test.jpg", model, CLASS_NAMES)

## ROC-AUC

In [ ]:
val_gen.reset()
y_true = val_data.classes
y_prob = model.predict(val_data, verbose=1)

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc as auc_score

y_bin     = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
macro_auc = roc_auc_score(y_bin, y_prob, average='macro', multi_class='ovr')
print(f"Macro ROC-AUC: {macro_auc:.4f}")

plt.figure(figsize=(8, 6))
for i in range(min(NUM_CLASSES, 10)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
    plt.plot(fpr, tpr, lw=1.5,
             label=f"{CLASS_NAMES[i]} (AUC={auc_score(fpr, tpr):.2f})")

plt.plot([0, 1], [0, 1], "k--", lw=1)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves — ConvNeXtBase")
plt.legend(fontsize=7, loc="lower right")
plt.tight_layout()
plt.show()